# Explore `avaeziaiteam/harakat-dataset`

This is a **private** HF dataset repo, so you need to authenticate first.

1. Create/copy a HF access token (read scope is enough) from https://huggingface.co/settings/tokens
2. Either run `huggingface-cli login` in a terminal, or log in inline below (uncomment).

In [ ]:
from huggingface_hub import login
login()

In [12]:
from datasets import load_dataset

ds = load_dataset("avaeziaiteam/harakat-dataset")
ds

DatasetDict({
    train: Dataset({
        features: ['raw', 'harakat'],
        num_rows: 7182
    })
})

In [15]:
ds["train"][0]

{'raw': 'به همین دلیل هرگز با این ازدواج موافقت نخواهیم کرد سودابه روی از پنجره برگردانید',
 'harakat': 'بِه هَمینْ دَلیلْ هَرْگِزْ با اینْ اِزْدِواجْ موافقت نَخواهیمْ کَرْدْ سودابِه رویْ اَزْ پَنْجَرِه بَرْگَرْدانیدْ'}

In [9]:
split = list(ds.keys())[0]
df = ds[split].to_pandas()
df.head()

,raw,harakat
0,به همین دلیل هرگز با این ازدواج موافقت نخواهیم...,بِه هَمینْ دَلیلْ هَرْگِزْ با اینْ اِزْدِواجْ ...
1,چشمان مادر یک‌لحظه از وحشت و درد گشاد شدند نگا...,چَشْمانِ مادَرْ یِکْ‌لَحْظِه اَزْ وَحْشَتْ وَ ...
2,پدر و مادر بیچاره سودابه حتی جرئت نداشتند تا د...,پِدَرْ وَ مادَرِ بیچاره سودابِه حتی جُرْئَتْ ن...
3,گوهری که می‌خواست به دامان خس بغلتد واقعا که ا...,گَوْهَری کِه می‌خواسْتْ بِه دامانِ خس بِغَلْتَ...
4,تابلوهای نقاشی که دیوارها را زینت می‌دادند همه...,تابْلوهایِ نَقّاشی کِه دیوارها را زینَتْ می‌دا...


In [11]:
len(df["raw"].tolist()), len(set(df["raw"].tolist()))

(7182, 7180)

In [ ]:
df.shape, df.columns.tolist()

## Harakat (diacritic) counts in the `harakat` column

Counts how many Fatha, Damma, Kasra, and Sokun marks appear across all rows of the `harakat` column, so we can see whether each diacritic occurrence should be treated as one training example (i.e. per-character label) rather than counting rows.

In [6]:
from collections import Counter

HARAKAT = {
    "Fatha": "َ",
    "Damma": "ُ",
    "Kasra": "ِ",
    "Sokun": "ْ",
}

counts = Counter()
for text in df["harakat"]:
    for name, char in HARAKAT.items():
        counts[name] += text.count(char)

counts

Counter({'Sokun': 221141, 'Fatha': 165895, 'Kasra': 106192, 'Damma': 30926})

In [7]:
# Context: rows vs. total diacritic occurrences.
# If training is per-character (as in train_diacritizer.py, which labels each
# base character NONE/KASRA/SOKUN), then each occurrence below is one labeled
# character, not each row -> the real example count is close to the number of
# base characters, not len(df).
num_rows = len(df)
total_diacritics = sum(counts.values())
total_chars = df["harakat"].str.len().sum()

print(f"rows: {num_rows}")
print(f"total diacritic marks: {total_diacritics}")
print(f"total characters (incl. diacritics): {total_chars}")
print(f"avg diacritics per row: {total_diacritics / num_rows:.2f}")

rows: 7182
total diacritic marks: 524154
total characters (incl. diacritics): 2069658
avg diacritics per row: 72.98


## Rows with no diacritics at all

Check whether any row's `harakat` value contains none of the four marks (i.e. would train on an all-`NONE` label sequence).

In [8]:
has_any_harakat = df["harakat"].apply(lambda t: any(c in HARAKAT.values() for c in t))
no_harakat_rows = df[~has_any_harakat]

print(f"rows with NO diacritics: {len(no_harakat_rows)} / {len(df)}")
no_harakat_rows.head(10)

rows with NO diacritics: 0 / 7182


,raw,harakat
